In [ ]:
# =============================
# 1. IMPORTS
# =============================
import os
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import random

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", DEVICE)

# =============================
# 2. CONFIG
# =============================
SR = 22050
DURATION = 5
SAMPLES = SR * DURATION
N_MELS = 128
BATCH_SIZE = 16
EPOCHS = 15
NUM_CLASSES = 10

GENRES = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']

DATA_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
TRAIN_PATH = f"{DATA_PATH}/genres_stems"
TEST_PATH = f"{DATA_PATH}/mashups"

test_df = pd.read_csv(f"{DATA_PATH}/test.csv")

# =============================
# 3. BUILD TRAIN DATAFRAME
# =============================
rows = []

for genre in GENRES:
    genre_dir = os.path.join(TRAIN_PATH, genre)

    for song in os.listdir(genre_dir):
        song_path = os.path.join(genre_dir, song)

        if os.path.isdir(song_path):
            rows.append({
                "path": song_path,
                "label": GENRES.index(genre)
            })

train_df = pd.DataFrame(rows)

# =============================
# 4. AUDIO FUNCTIONS
# =============================

def load_audio(path):
    audio, _ = librosa.load(path, sr=SR)
    return audio


def mix_stems(song_path):

    stems = ['drums.wav','vocals.wav','bass.wav','others.wav']
    audio = 0

    for stem in stems:
        p = os.path.join(song_path, stem)
        if os.path.exists(p):
            audio += load_audio(p)

    audio = audio / (np.max(np.abs(audio)) + 1e-6)
    return audio


def random_crop(audio):

    if len(audio) < SAMPLES:
        audio = np.pad(audio,(0,SAMPLES-len(audio)))

    start = random.randint(0,len(audio)-SAMPLES)
    return audio[start:start+SAMPLES]


def add_noise(audio):
    noise = np.random.randn(len(audio))
    return audio + random.uniform(0.003,0.02)*noise


def pitch_shift(audio):
    return librosa.effects.pitch_shift(audio, sr=SR, n_steps=random.randint(-2,2))


def time_shift(audio):
    shift = int(0.2*SR)
    return np.roll(audio, shift)


def to_mel(audio):

    mel = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=N_MELS)
    mel = librosa.power_to_db(mel)

    mel = (mel - mel.mean())/(mel.std()+1e-6)

    return mel


# =============================
# 5. DATASET
# =============================

class MusicDataset(Dataset):

    def __init__(self, df, train=True):
        self.df = df
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]
        label = row['label']

        audio = mix_stems(row['path'])
        audio = random_crop(audio)

        if self.train:
            if random.random() < 0.5:
                audio = add_noise(audio)

            if random.random() < 0.5:
                audio = pitch_shift(audio)

            if random.random() < 0.5:
                audio = time_shift(audio)

        mel = to_mel(audio)

        mel = torch.tensor(mel).unsqueeze(0).float()

        return mel, label


# =============================
# 6. MODEL
# =============================

class CRNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(

            nn.Conv2d(1,16,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.lstm = nn.LSTM(32*32,128,batch_first=True,bidirectional=True)

        self.fc = nn.Linear(256,NUM_CLASSES)

    def forward(self,x):

        x = self.cnn(x)

        b,c,h,w = x.shape
        x = x.view(b,w,c*h)

        x,_ = self.lstm(x)

        x = x.mean(dim=1)

        return self.fc(x)


# =============================
# 7. TRAIN FUNCTION
# =============================

def train_fn(model,loader,optimizer,loss_fn):

    model.train()
    total_loss = 0

    for x,y in loader:

        x,y = x.to(DEVICE),y.to(DEVICE)

        optimizer.zero_grad()

        out = model(x)

        loss = loss_fn(out,y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss/len(loader)


# =============================
# 8. DATA LOADERS
# =============================

train_ds = MusicDataset(train_df,train=True)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)


# =============================
# 9. TRAIN LOOP
# =============================

model = CRNN().to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(),lr=1e-3)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

for epoch in range(EPOCHS):

    loss = train_fn(model,train_loader,optimizer,loss_fn)

    print(f"Epoch {epoch+1} Loss {loss:.4f}")


# =============================
# 10. PREDICT
# =============================

def predict(model,path):

    audio = load_audio(path)

    preds = []

    for _ in range(10):

        chunk = random_crop(audio)

        mel = to_mel(chunk)

        mel = torch.tensor(mel).unsqueeze(0).unsqueeze(0).float().to(DEVICE)

        with torch.no_grad():

            out = model(mel)

            preds.append(torch.softmax(out,dim=1).cpu().numpy())

    return np.mean(preds,axis=0)


# =============================
# 11. SAVE MODEL
# =============================

torch.save(model.state_dict(),"model.pth")


# =============================
# 12. CREATE SUBMISSION
# =============================

model.eval()

predictions = []

file_col = test_df.columns[1]

for file in test_df[file_col]:

    path = os.path.join(TEST_PATH, file)

    probs = predict(model, path)

    pred = GENRES[np.argmax(probs)]

    predictions.append(pred)

submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": predictions
})

submission.to_csv("submission.csv",index=False)

print("submission.csv saved")